# Prompting baseline — zero-shot, Qwen2.5-Coder-7B-Instruct (4-bit)

Đánh giá baseline zero-shot (không few-shot, không fine-tune) trên 32 mẫu `sample_dataset` (có nhãn),
so sánh với baseline tầm thường (constant prediction) để biết model có học được tín hiệu thật hay không.

Model: `Qwen/Qwen2.5-Coder-7B-Instruct`, 4-bit NF4 (bitsandbytes), greedy decoding (seed=42).
Xem [METHOD.md](../METHOD.md) để biết chi tiết prompt/pipeline.

In [1]:
import sys, json
sys.path.insert(0, "../src")
from data import load_sample_dataset
from eval import task1_metrics, task2_metrics, task3_compliance

ds = load_sample_dataset()
label_space = ds.label_space["taxonomy_error"]

def load_preds(task_num):
    with open(f"../outputs/predictions_task{task_num}_sample.json", encoding="utf-8") as f:
        return json.load(f)

preds1 = load_preds(1)
preds2 = load_preds(2)
preds3 = load_preds(3)
print(len(preds1), len(preds2), len(preds3), "predictions loaded")


32 32 32 predictions loaded


## Task 1 — QWK / MAE / exact-match, so với baseline hằng số

In [2]:
pred1_by_id = {p["sample_id"]: p["output"] for p in preds1}
m1 = task1_metrics(ds.tasks[1], pred1_by_id)
print("zero-shot:", {k: round(v, 3) if isinstance(v, float) else v for k, v in m1.items() if k != "exact_match"})
print("  exact_match per dim:", m1["exact_match"])

scores = [s["output"]["total_score"] for s in ds.tasks[1]]
mean_c, mode_c = round(sum(scores) / len(scores)), max(set(scores), key=scores.count)
for name, const in [("mean", mean_c), ("mode", mode_c)]:
    pred_c = {s["sample_id"]: {"rubric": s["output"]["rubric"], "total_score": const} for s in ds.tasks[1]}
    mc = task1_metrics(ds.tasks[1], pred_c)
    print(f"constant={name}({const}): QWK={mc['qwk']:.3f} MAE={mc['mae']:.3f}")


zero-shot: {'n': 32, 'qwk': 0.393, 'mae': 3.188}
  exact_match per dim: {'compilable': 0.71875, 'io_format': 0.625, 'logic': 0.375, 'edge_case': 0.5, 'complexity': 0.5625, 'code_quality': 0.5}
constant=mean(5): QWK=0.000 MAE=3.469
constant=mode(7): QWK=0.000 MAE=3.156


**Đọc kết quả:** zero-shot có QWK > 0 rõ rệt trong khi baseline hằng số QWK = 0 (không có biến thiên) —
model có học được *thứ tự* điểm số thật. Nhưng MAE zero-shot xấp xỉ baseline hằng số → model **lệch chuẩn (calibration)**
so với thang điểm giáo viên, dù đã nắm được tương quan. Đây là mục tiêu chính cho few-shot: cho model thấy ví dụ
điểm số thật để canh thang, không phải để dạy cách đọc code.

## Phân tích lỗi Task 1 — các mẫu lệch nhiều nhất

In [3]:
rows = []
for s in ds.tasks[1]:
    sid = s["sample_id"]
    gold_total = s["output"]["total_score"]
    pred_total = pred1_by_id[sid]["total_score"]
    rows.append((sid, gold_total, pred_total, abs(gold_total - pred_total)))
rows.sort(key=lambda r: -r[3])
print(f"{'sample_id':<15}{'gold':>6}{'pred':>6}{'|diff|':>8}")
for sid, g, p, d in rows[:10]:
    print(f"{sid:<15}{g:>6}{p:>6}{d:>8}")


sample_id        gold  pred  |diff|
EX01-S104           9     0       9
EX01-S112           9     0       9
EX01-S101           8     0       8
EX01-S110           7     0       7
EX02-S208           7     0       7
EX02-S214          10     3       7
EX01-S102           9     3       6
EX01-S107           2     8       6
EX01-S108           6     0       6
EX02-S205           9     4       5


## Task 2 — macro/micro-F1, so với baseline hằng số

In [4]:
pred2_by_id = {p["sample_id"]: p["output"]["taxonomy_error"] for p in preds2}
m2 = task2_metrics(ds.tasks[2], pred2_by_id, label_space)
print("zero-shot: macro_f1=%.3f micro_f1=%.3f" % (m2["macro_f1"], m2["micro_f1"]))
print("per-label F1:", {k: round(v, 2) for k, v in m2["per_label_f1"].items()})

for name, const in [("empty", []), ("most_common", ["Lỗi logic"])]:
    pred_c = {s["sample_id"]: const for s in ds.tasks[2]}
    mc = task2_metrics(ds.tasks[2], pred_c, label_space)
    print(f"constant={name}: macro_f1={mc['macro_f1']:.3f} micro_f1={mc['micro_f1']:.3f}")


zero-shot: macro_f1=0.183 micro_f1=0.286
per-label F1: {'Lỗi biên dịch': 0.32, 'Lỗi nhập/xuất': 0.0, 'Lỗi logic': 0.52, 'Lỗi vòng lặp': 0.0, 'Lỗi mảng/chuỗi': 1.0, 'Lỗi hàm': 0.0, 'Lỗi edge case': 0.0, 'Lỗi thuật toán': 0.0, 'Lỗi hard-code': 0.0, 'Lỗi style': 0.0}
constant=empty: macro_f1=0.000 micro_f1=0.000
constant=most_common: macro_f1=0.055 micro_f1=0.329


**Đọc kết quả:** zero-shot vượt baseline hằng số trên macro-F1 (metric chính) nhưng thấp hơn baseline
"luôn đoán nhãn phổ biến nhất" trên micro-F1 — model dự đoán rải đều nhiều nhãn hơn (đúng hướng cho macro-F1,
vì macro-F1 phạt nặng việc chỉ đoán nhãn phổ biến) nhưng độ chính xác per-label vẫn còn thấp.

## Task 3 — Level compliance trên 32 mẫu

In [5]:
pred3_by_id = {p["sample_id"]: p["output"]["feedback"] for p in preds3}
level_by_id = {s["sample_id"]: s["input"]["target_feedback_level"] for s in ds.tasks[3]}
triples = [(sid, level_by_id[sid], fb) for sid, fb in pred3_by_id.items()]
m3 = task3_compliance(triples)
print("compliance_rate:", m3["compliance_rate"], "violations:", m3["violations"])

# qualitative: compare one Level 1 and one Level 3 feedback, model vs teacher
sample_l1 = next(s for s in ds.tasks[3] if s["input"]["target_feedback_level"].startswith("Level 1"))
sample_l3 = next(s for s in ds.tasks[3] if s["input"]["target_feedback_level"].startswith("Level 3"))
for s in [sample_l1, sample_l3]:
    sid = s["sample_id"]
    print(f"\n--- {sid} ({s['input']['target_feedback_level']}) ---")
    print("teacher :", s["output"]["feedback"])
    print("model   :", pred3_by_id[sid][:300])


compliance_rate: 1.0 violations: []

--- EX02-S201 (Level 1 - Gợi ý nhẹ) ---
teacher : Tốt.
model   : Đúng rồi! Bạn đã viết chương trình kiểm tra số Armstrong một cách chính xác. Chúc mừng bạn!

--- EX02-S203 (Level 3 - Gợi ý sửa) ---
teacher : Lỗi biên dịch. Em nên chú ý học kỹ cú pháp và làm bài tập nhiều hơn nhaa.
model   : Đề bài yêu cầu bạn kiểm tra xem một số có phải là số Armstrong hay không. Tuy nhiên, mã bạn cung cấp chứa nhiều lỗi. Đầu tiên, hàm `ts` không được sử dụng đúng cách. Thứ hai, hàm `sum` chưa được định nghĩa. Cuối cùng, trong hàm `main`, bạn đang gọi hàm `sum` nhưng không truyền đối số nào. Bạn cũng c


**Đọc kết quả:** compliance = 1.0 trên 32 mẫu — checker rule-based (regex tìm code/markers) không bắt được
vi phạm nào, nhưng đây là một heuristic thô: nó chỉ chặn code/markup rõ ràng, chưa chặn được việc "gợi ý cách sửa"
bằng lời (ví dụ "nên dùng vector thay vì mảng động" ở Level 2 — về lý thuyết đã hơi vượt mức "chỉ chẩn đoán").
Cần tinh chỉnh checker hoặc đánh giá thủ công một phần trong báo cáo cuối.

## Lưu ý quan trọng: Task 3 trên public_test khác sample_dataset

`sample_dataset` cho `taxonomy_error` sẵn ở input (output chỉ cần `feedback`), nhưng `public_test_release`
**không** cho nhãn lỗi — model phải tự chẩn đoán (`output = {taxonomy_error, feedback}`). Baseline ở notebook này
chỉ đo được nhánh "nhãn cho sẵn"; nhánh tự chẩn đoán (dùng cho submission thật) chưa có ground truth để so sánh
trực tiếp — xem [METHOD.md](../METHOD.md) mục "Task 3 schema fork".

## Tóm tắt & bước tiếp theo

| Task | Metric chính | Zero-shot | Baseline hằng số |
|---|---|---|---|
| 1 | QWK | xem trên | 0.000 |
| 1 | MAE | xem trên | ~3.2–3.5 |
| 2 | macro-F1 | xem trên | 0.055 (nhãn phổ biến) |
| 3 | level compliance | xem trên | — |

Zero-shot vượt baseline hằng số trên các metric chính (QWK, macro-F1) nhưng MAE còn kém → cần **few-shot**
(cho ví dụ điểm số thật để canh thang) trước khi fine-tune. Đây là bước tiếp theo trong [METHOD.md](../METHOD.md).